# Spark SQL and DataFrames: Introduction to Built-in Data Sources

## Using Spark SQL in Spark Applications

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession
  .builder
  .appName("SparkSQLExampleApp")
  .getOrCreate())

# read data set into temporary view
csv_file = "../data/learning-spark-v2/flights/departuredelays.csv"
df = (spark.read.format("csv")
  .option("inferSchema", "true")
  .option("header", "true")
  .load(csv_file))
df.createOrReplaceTempView("us_delay_flights_tbl")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 03:19:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
_sql = """SELECT distance, origin, destination
FROM us_delay_flights_tbl WHERE distance > 1000
ORDER BY distance DESC"""
spark.sql(_sql).show(10)

+--------+------+-----------+
|distance|origin|destination|
+--------+------+-----------+
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
+--------+------+-----------+
only showing top 10 rows



In [4]:
_sql = """SELECT date, delay, origin, destination
FROM us_delay_flights_tbl
WHERE delay > 120 AND ORIGIN = 'SFO' AND DESTINATION = 'ORD'
ORDER by delay DESC"""
spark.sql(_sql).show(10)

+-------+-----+------+-----------+
|   date|delay|origin|destination|
+-------+-----+------+-----------+
|2190925| 1638|   SFO|        ORD|
|1031755|  396|   SFO|        ORD|
|1022330|  326|   SFO|        ORD|
|1051205|  320|   SFO|        ORD|
|1190925|  297|   SFO|        ORD|
|2171115|  296|   SFO|        ORD|
|1071040|  279|   SFO|        ORD|
|1051550|  274|   SFO|        ORD|
|3120730|  266|   SFO|        ORD|
|1261104|  258|   SFO|        ORD|
+-------+-----+------+-----------+
only showing top 10 rows



In [5]:
_sql = """SELECT delay, origin, destination,
CASE
  WHEN delay > 360 THEN 'Very Long Delays'
  WHEN delay > 120 AND delay < 360 THEN 'Long Delays'
  WHEN delay > 60 AND delay < 120 THEN 'Short Delays'
  WHEN delay > 0 and delay < 60 THEN 'Tolerable Delays'
  WHEN delay = 0 THEN 'No Delays'
  ELSE 'Early'
END AS Flight_Delays
FROM us_delay_flights_tbl
ORDER BY origin, delay DESC"""
spark.sql(_sql).show(10)

+-----+------+-----------+-------------+
|delay|origin|destination|Flight_Delays|
+-----+------+-----------+-------------+
|  333|   ABE|        ATL|  Long Delays|
|  305|   ABE|        ATL|  Long Delays|
|  275|   ABE|        ATL|  Long Delays|
|  257|   ABE|        ATL|  Long Delays|
|  247|   ABE|        ATL|  Long Delays|
|  247|   ABE|        DTW|  Long Delays|
|  219|   ABE|        ORD|  Long Delays|
|  211|   ABE|        ATL|  Long Delays|
|  197|   ABE|        DTW|  Long Delays|
|  192|   ABE|        ORD|  Long Delays|
+-----+------+-----------+-------------+
only showing top 10 rows



In [6]:
# in DataFrame API
from pyspark.sql.functions import col, desc

(df.select("distance", "origin", "destination")
  .where(col("distance") > 1000)
  .orderBy(desc("distance"))).show(10)

+--------+------+-----------+
|distance|origin|destination|
+--------+------+-----------+
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
+--------+------+-----------+
only showing top 10 rows



## SQL Tables and Views

In [7]:
spark.conf.get('spark.sql.warehouse.dir')

'file:/workspaces/learning-cloudnative/codes/compute/spark/learning-spark/spark-warehouse'

In [8]:
# Creating SQL Databases and Tables
spark.sql("CREATE DATABASE learn_spark_db") # default database: `default`
spark.sql("USE learn_spark_db")

DataFrame[]

In [10]:
# Creating a managed table
# ERROR: 26/04/21 03:28:50 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
spark.conf.set('spark.sql.legacy.createHiveTableByDefault', False)
spark.sql("CREATE TABLE managed_us_delay_flights_tbl (date STRING, delay INT, distance INT, origin STRING, destination STRING)")

# or in DataFrame API
# df.write.saveAsTable("managed_us_delay_flights_tbl")

DataFrame[]

In [15]:
# Creating an unmanaged table

# spark.sql("DROP TABLE us_delay_flights_tbl")
spark.sql("""CREATE TABLE us_delay_flights_tbl(date STRING, delay INT,
distance INT, origin STRING, destination STRING)
USING csv OPTIONS (PATH '/workspaces/learning-cloudnative/codes/compute/spark/data/learning-spark-v2/flights/departuredelays.csv')""")

DataFrame[]

In [17]:
spark.sql("SELECT * FROM us_delay_flights_tbl").show(10)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|    date| NULL|    NULL|origin|destination|
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
|01020605|   -4|     602|   ABE|        ATL|
|01031245|   -4|     602|   ABE|        ATL|
|01030605|    0|     602|   ABE|        ATL|
|01041243|   10|     602|   ABE|        ATL|
|01040605|   28|     602|   ABE|        ATL|
|01051245|   88|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 10 rows



In [24]:
# Creating Views

sql_sfo = """CREATE OR REPLACE GLOBAL TEMP VIEW us_origin_airport_SFO_global_tmp_view AS
SELECT date, delay, origin, destination from us_delay_flights_tbl WHERE
origin = 'SFO'"""
sql_jfk="""CREATE OR REPLACE TEMP VIEW us_origin_airport_JFK_tmp_view AS
SELECT date, delay, origin, destination from us_delay_flights_tbl WHERE
origin = 'JFK'"""
spark.sql(sql_sfo)
spark.sql(sql_jfk)
spark.sql("SELECT * FROM global_temp.us_origin_airport_SFO_global_tmp_view").show(2)
spark.sql("SELECT * FROM us_origin_airport_JFK_tmp_view").show(2)

# use DataFrame API
# df_sfo = spark.sql("SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'SFO'")
# df_jfk = spark.sql("SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'JFK'")
# # temporary and global temporary view
# df_sfo.createOrReplaceGlobalTempView("us_origin_airport_SFO_global_tmp_view")
# df_jfk.createOrReplaceTempView("us_origin_airport_JFK_tmp_view")


# spark.sql("DROP VIEW IF EXISTS us_origin_airport_SFO_global_tmp_view")
# spark.sql("DROP VIEW IF EXISTS us_origin_airport_JFK_tmp_view")
# use DataFrame API
spark.catalog.dropGlobalTempView("us_origin_airport_SFO_global_tmp_view")
spark.catalog.dropTempView("us_origin_airport_JFK_tmp_view")

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|01011250|   55|   SFO|        JFK|
|01012230|    0|   SFO|        JFK|
+--------+-----+------+-----------+
only showing top 2 rows

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|01010900|   14|   JFK|        LAX|
|01011200|   -3|   JFK|        LAX|
+--------+-----+------+-----------+
only showing top 2 rows



True

In [26]:
# Viewing the Metadata
spark.catalog.listDatabases(), \
spark.catalog.listTables(), \
spark.catalog.listColumns("us_delay_flights_tbl")

([Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/workspaces/learning-cloudnative/codes/compute/spark/learning-spark/spark-warehouse'),
  Database(name='learn_spark_db', catalog='spark_catalog', description='', locationUri='file:/workspaces/learning-cloudnative/codes/compute/spark/learning-spark/spark-warehouse/learn_spark_db.db')],
 [Table(name='managed_us_delay_flights_tbl', catalog='spark_catalog', namespace=['learn_spark_db'], description=None, tableType='MANAGED', isTemporary=False),
  Table(name='us_delay_flights_tbl', catalog='spark_catalog', namespace=['learn_spark_db'], description=None, tableType='EXTERNAL', isTemporary=False)],
 [Column(name='date', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
  Column(name='delay', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
  Column(name='distance', description=None, dataType='int', nullable=True, isParti

In [ ]:
# Caching SQL Tables
"""
CACHE [LAZY] TABLE <table-name>
UNCACHE TABLE <table-name>
"""

In [27]:
# Reading Tables into DataFrames
us_flights_df = spark.sql("SELECT * FROM us_delay_flights_tbl")
us_flights_df.show(2)
us_flights_df2 = spark.table("us_delay_flights_tbl")
us_flights_df2.show(2)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|    date| NULL|    NULL|origin|destination|
|01011245|    6|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 2 rows

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|    date| NULL|    NULL|origin|destination|
|01011245|    6|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 2 rows



## Data Sources for DataFrames and SQL Tables

- DataFrameReader
  - parquet: learning-spark-v2/flights/summarydata/parquet/2010-summary.parquet
  - CSV: learning-spark-v2/flights/summary-data/csv/*
  - JSON: learning-spark-v2/flights/summary-data/json/*
- DataFrameWriter
- Parquet: learning-spark-v2/flights/summary-data/parquet/2010-summary.parquet/
  - Reading Parquet files into a Spark SQL table
  - Writing DataFrames to Parquet files
  - Writing DataFrames to Spark SQL tables
- JSON: learning-spark-v2/flights/summary-data/json/*
  - Reading a JSON file into a Spark SQL table
  - Writing DataFrames to JSON files
  - JSON data source options
- CSV: learning-spark-v2/flights/summary-data/csv/*
  - Reading a CSV file into a DataFrame
  - Reading a CSV file into a Spark SQL table
  - Writing DataFrames to CSV files
  - CSV data source options
- Avro: learning-spark-v2/flights/summary-data/avro/*
  - Reading an Avro file into a DataFrame
  - Reading an Avro file into a Spark SQL table
  - Writing DataFrames to Avro files
  - Avro data source options
- ORC: learning-spark-v2/flights/summary-data/orc/*
  - Apache ORC is a columnar format which has more advanced features like native zstd compression, bloom filter and columnar encryption.
  - Reading an ORC file into a DataFrame
  - Reading an ORC file into a Spark SQL table
  - Writing DataFrames to ORC files
- Images: learning-spark-v2/cctvVideos/train_images/
  - Reading an image file into a DataFrame
- Binary Files: learning-spark-v2/cctvVideos/train_images/
  - Reading a binary file into a DataFrame

# Spark SQL and DataFrames: Interacting with External Data Sources

## Spark SQL and Apache Hive

In [28]:
# User-Defined Functions
from pyspark.sql.types import LongType

def cubed(s):
  return s * s * s

spark.udf.register("cubed", cubed, LongType())
spark.range(1, 9).createOrReplaceTempView("udf_test")
spark.sql("SELECT id, cubed(id) AS id_cubed FROM udf_test").show()

+---+--------+
| id|id_cubed|
+---+--------+
|  1|       1|
|  2|       8|
|  3|      27|
|  4|      64|
|  5|     125|
|  6|     216|
|  7|     343|
|  8|     512|
+---+--------+



In [30]:
# Pandas UDFs: use Apache Arrow - pip install pyarrow
import pandas as pd

from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import LongType

def cubed(a: pd.Series) -> pd.Series:
  return a * a * a

cubed_udf = pandas_udf(cubed, returnType=LongType())

x = pd.Series([1, 2, 3])
cubed(x)

df = spark.range(1, 4)
df.select("id", cubed_udf(col("id"))).show()

+---+---------+
| id|cubed(id)|
+---+---------+
|  1|        1|
|  2|        8|
|  3|       27|
+---+---------+



## Querying with the Spark SQL Shell, Beeline, and Tableau

- spark-sql
- Beeline
- Tableau Destop 2019.2

## External Data Sources

- JDBC: `./bin/spark-shell --driver-class-path $database.jar --jars $database.jar`
- PostgreSQL: `bin/spark-shell --jars postgresql-42.2.6.jar`
- MySQL: `bin/spark-shell --jars mysql-connector-java_8.0.16-bin.jar`
- Azure Cosmos DB
- MS SQL Server: `bin/spark-shell --jars mssql-jdbc-7.2.2.jre8.jar`
- Other External Sources
  - Apache Cassandra
  - Snowflake
  - MongoDB

## Higher-Order Functions in DataFrames and Spark SQL

utility functions
```python
get_json_object()
from_json()
to_json()
explode()
selectExpr()
```


In [33]:
# Built-in Functions for Complex Data Types
## Array type functions
spark.sql("SELECT array_distinct(array(1, 2, 3, null, 3))").show()

## Map functions
spark.sql("SELECT map_from_arrays(array(1.0, 3.0), array('2', '4'))").show()

+---------------------------------------+
|array_distinct(array(1, 2, 3, NULL, 3))|
+---------------------------------------+
|                        [1, 2, 3, NULL]|
+---------------------------------------+

+---------------------------------------------+
|map_from_arrays(array(1.0, 3.0), array(2, 4))|
+---------------------------------------------+
|                         {1.0 -> 2, 3.0 -> 4}|
+---------------------------------------------+



In [34]:
# Higher-Order Functions

from pyspark.sql.types import *
schema = StructType([StructField("celsius", ArrayType(IntegerType()))])
t_list = [[35, 36, 32, 30, 40, 42, 38]], [[31, 32, 34, 55, 56]]
t_c = spark.createDataFrame(t_list, schema)
t_c.createOrReplaceTempView("tC")
# Show the DataFrame
t_c.show()

+--------------------+
|             celsius|
+--------------------+
|[35, 36, 32, 30, ...|
|[31, 32, 34, 55, 56]|
+--------------------+



In [36]:
## transform
spark.sql("""SELECT celsius, transform(celsius, t -> ((t * 9) div 5) + 32) as fahrenheit FROM tC""").show()
## filter
spark.sql("""SELECT celsius, filter(celsius, t -> t > 38) as high FROM tC""").show()
## exists
spark.sql("""SELECT celsius, exists(celsius, t -> t = 38) as threshold FROM tC""").show()
## reduce
spark.sql("""SELECT celsius,
reduce(
  celsius,
  0,
  (t, acc) -> t + acc,
  acc -> (acc div size(celsius) * 9 div 5) + 32
) as avgFahrenheit
FROM tC
""").show()

+--------------------+--------------------+
|             celsius|          fahrenheit|
+--------------------+--------------------+
|[35, 36, 32, 30, ...|[95, 96, 89, 86, ...|
|[31, 32, 34, 55, 56]|[87, 89, 93, 131,...|
+--------------------+--------------------+

+--------------------+--------+
|             celsius|    high|
+--------------------+--------+
|[35, 36, 32, 30, ...|[40, 42]|
|[31, 32, 34, 55, 56]|[55, 56]|
+--------------------+--------+

+--------------------+---------+
|             celsius|threshold|
+--------------------+---------+
|[35, 36, 32, 30, ...|     true|
|[31, 32, 34, 55, 56]|    false|
+--------------------+---------+

+--------------------+-------------+
|             celsius|avgFahrenheit|
+--------------------+-------------+
|[35, 36, 32, 30, ...|           96|
|[31, 32, 34, 55, 56]|          105|
+--------------------+-------------+



## Common DataFrames and Spark SQL Operations

- Aggregate functions
- Collection functions
- Datetime functions
- Math functions
- Miscellaneous functions
- Non-aggregate functions
- Sorting functions
- String functions
- UDF functions
- Window functions              <---
- Unions and joins              <---
- Modifications                 <---

In [37]:
from pyspark.sql.functions import expr

tripdelaysFilePath ="../data/learning-spark-v2/flights/departuredelays.csv"
airportsnaFilePath = "../data/learning-spark-v2/flights/airport-codes-na.txt"

airportsna = (spark.read
  .format("csv")
  .options(header="true", inferSchema="true", sep="\t")
  .load(airportsnaFilePath))
airportsna.createOrReplaceTempView("airports_na")

departureDelays = (spark.read
  .format("csv")
  .options(header="true")
  .load(tripdelaysFilePath))
departureDelays = (departureDelays
  .withColumn("delay", expr("CAST(delay as INT) as delay"))
  .withColumn("distance", expr("CAST(distance as INT) as distance")))
departureDelays.createOrReplaceTempView("departureDelays")

foo = (departureDelays
  .filter(expr("""origin == 'SEA' and destination == 'SFO' and date like '01010%' and delay > 0""")))
foo.createOrReplaceTempView("foo")

In [39]:
spark.sql("SELECT * FROM airports_na LIMIT 10").show()
spark.sql("SELECT * FROM departureDelays LIMIT 10").show()
spark.sql("SELECT * FROM foo").show()

+-----------+-----+-------+----+
|       City|State|Country|IATA|
+-----------+-----+-------+----+
| Abbotsford|   BC| Canada| YXX|
|   Aberdeen|   SD|    USA| ABR|
|    Abilene|   TX|    USA| ABI|
|      Akron|   OH|    USA| CAK|
|    Alamosa|   CO|    USA| ALS|
|     Albany|   GA|    USA| ABY|
|     Albany|   NY|    USA| ALB|
|Albuquerque|   NM|    USA| ABQ|
| Alexandria|   LA|    USA| AEX|
|  Allentown|   PA|    USA| ABE|
+-----------+-----+-------+----+

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
|01020605|   -4|     602|   ABE|        ATL|
|01031245|   -4|     602|   ABE|        ATL|
|01030605|    0|     602|   ABE|        ATL|
|01041243|   10|     602|   ABE|        ATL|
|01040605|   28|     602|   ABE|        ATL|
|01051245|   88|     602|   ABE|        AT

In [41]:
# Unions
bar = departureDelays.union(foo)
bar.createOrReplaceTempView("bar")
bar.filter(expr("""origin == 'SEA' AND destination == 'SFO' AND date LIKE '01010%' AND delay > 0""")).show()

spark.sql("""
SELECT *
FROM bar
WHERE origin = 'SEA'
  AND destination = 'SFO'
  AND date LIKE '01010%'
  AND delay > 0
""").show()

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+



In [43]:
# Joins
foo.join(
  airportsna,
  airportsna.IATA == foo.origin
).select("City", "State", "date", "delay", "distance", "destination").show()

spark.sql("""
SELECT a.City, a.State, f.date, f.delay, f.distance, f.destination
FROM foo f
JOIN airports_na a
  ON a.IATA = f.origin
""").show()

+-------+-----+--------+-----+--------+-----------+
|   City|State|    date|delay|distance|destination|
+-------+-----+--------+-----+--------+-----------+
|Seattle|   WA|01010710|   31|     590|        SFO|
|Seattle|   WA|01010955|  104|     590|        SFO|
|Seattle|   WA|01010730|    5|     590|        SFO|
+-------+-----+--------+-----+--------+-----------+

+-------+-----+--------+-----+--------+-----------+
|   City|State|    date|delay|distance|destination|
+-------+-----+--------+-----+--------+-----------+
|Seattle|   WA|01010710|   31|     590|        SFO|
|Seattle|   WA|01010955|  104|     590|        SFO|
|Seattle|   WA|01010730|    5|     590|        SFO|
+-------+-----+--------+-----+--------+-----------+



In [46]:
# Windowing

spark.sql("""DROP TABLE IF EXISTS departureDelaysWindow""")
          
spark.sql("""CREATE TABLE departureDelaysWindow AS
SELECT origin, destination, SUM(delay) AS TotalDelays
FROM departureDelays
WHERE origin IN ('SEA', 'SFO', 'JFK')
  AND destination IN ('SEA', 'SFO', 'JFK', 'DEN', 'ORD', 'LAX', 'ATL')
GROUP BY origin, destination""")
          
spark.sql("""SELECT * FROM departureDelaysWindow""").show()

+------+-----------+-----------+
|origin|destination|TotalDelays|
+------+-----------+-----------+
|   JFK|        ORD|       5608|
|   JFK|        SFO|      35619|
|   JFK|        DEN|       4315|
|   JFK|        ATL|      12141|
|   JFK|        SEA|       7856|
|   JFK|        LAX|      35755|
|   SEA|        LAX|       9359|
|   SFO|        ORD|      27412|
|   SFO|        DEN|      18688|
|   SFO|        SEA|      17080|
|   SEA|        SFO|      22293|
|   SFO|        ATL|       5091|
|   SEA|        DEN|      13645|
|   SEA|        ATL|       4535|
|   SEA|        ORD|      10041|
|   SFO|        JFK|      24100|
|   SFO|        LAX|      40798|
|   SEA|        JFK|       4667|
+------+-----------+-----------+



In [ ]:
spark.sql("""
SELECT origin, destination, SUM(TotalDelays) -- AS TotalDelays
FROM departureDelaysWindow
-- WHERE origin = 'JFK'
GROUP BY origin, destination
ORDER BY SUM(TotalDelays) DESC
LIMIT 3
""").show()


spark.sql("""
SELECT origin, destination, TotalDelays, rank
FROM (
  SELECT origin, destination, TotalDelays, dense_rank()
  OVER (PARTITION BY origin ORDER BY TotalDelays DESC) as rank
  FROM departureDelaysWindow
) t
WHERE rank <= 3
""").show()

+------+-----------+----------------+
|origin|destination|sum(TotalDelays)|
+------+-----------+----------------+
|   SFO|        LAX|           40798|
|   JFK|        LAX|           35755|
|   JFK|        SFO|           35619|
+------+-----------+----------------+

+------+-----------+-----------+----+
|origin|destination|TotalDelays|rank|
+------+-----------+-----------+----+
|   JFK|        LAX|      35755|   1|
|   JFK|        SFO|      35619|   2|
|   JFK|        ATL|      12141|   3|
|   SEA|        SFO|      22293|   1|
|   SEA|        DEN|      13645|   2|
|   SEA|        ORD|      10041|   3|
|   SFO|        LAX|      40798|   1|
|   SFO|        ORD|      27412|   2|
|   SFO|        JFK|      24100|   3|
+------+-----------+-----------+----+



In [54]:
# Modifications
foo.show()

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+



In [55]:
## Adding new columns
from pyspark.sql.functions import expr
foo2 = (foo.withColumn(
    "status",
    expr("CASE WHEN delay <= 10 THEN 'On-time' ELSE 'Delayed' END")
  ))
foo2.show()

## Dropping columns
foo3 = foo2.drop("delay")
foo3.show()

## Renaming columns
foo4 = foo3.withColumnRenamed("status", "flight_status")
foo4.show()

+--------+-----+--------+------+-----------+-------+
|    date|delay|distance|origin|destination| status|
+--------+-----+--------+------+-----------+-------+
|01010710|   31|     590|   SEA|        SFO|Delayed|
|01010955|  104|     590|   SEA|        SFO|Delayed|
|01010730|    5|     590|   SEA|        SFO|On-time|
+--------+-----+--------+------+-----------+-------+

+--------+--------+------+-----------+-------+
|    date|distance|origin|destination| status|
+--------+--------+------+-----------+-------+
|01010710|     590|   SEA|        SFO|Delayed|
|01010955|     590|   SEA|        SFO|Delayed|
|01010730|     590|   SEA|        SFO|On-time|
+--------+--------+------+-----------+-------+

+--------+--------+------+-----------+-------------+
|    date|distance|origin|destination|flight_status|
+--------+--------+------+-----------+-------------+
|01010710|     590|   SEA|        SFO|      Delayed|
|01010955|     590|   SEA|        SFO|      Delayed|
|01010730|     590|   SEA|       

In [60]:
## Pivoting
spark.sql("""
SELECT destination, CAST(SUBSTRING(date, 0, 2) AS int) AS month, delay
FROM departureDelays
WHERE origin = 'SEA'
""").show(10)

spark.sql("""
SELECT * FROM (
  SELECT destination, CAST(SUBSTRING(date, 0, 2) AS int) AS month, delay
  FROM departureDelays
  WHERE origin = 'SEA'
)
PIVOT (
  CAST(AVG(delay) AS DECIMAL(4, 2)) AS AvgDelay, MAX(delay) AS MaxDelay
  FOR month IN (1 JAN, 2 FEB)
)
ORDER BY destination
""").show(10)


+-----------+-----+-----+
|destination|month|delay|
+-----------+-----+-----+
|        ORD|    1|   92|
|        JFK|    1|   -7|
|        DFW|    1|   -5|
|        MIA|    1|   -3|
|        DFW|    1|   -3|
|        DFW|    1|    1|
|        ORD|    1|  -10|
|        DFW|    1|   -6|
|        DFW|    1|   -2|
|        ORD|    1|   -3|
+-----------+-----+-----+
only showing top 10 rows

+-----------+------------+------------+------------+------------+
|destination|JAN_AvgDelay|JAN_MaxDelay|FEB_AvgDelay|FEB_MaxDelay|
+-----------+------------+------------+------------+------------+
|        ABQ|       19.86|         316|       11.42|          69|
|        ANC|        4.44|         149|        7.90|         141|
|        ATL|       11.98|         397|        7.73|         145|
|        AUS|        3.48|          50|       -0.21|          18|
|        BOS|        7.84|         110|       14.58|         152|
|        BUR|       -2.03|          56|       -1.89|          78|
|        CLE|   

# Spark SQL and Datasets

In [61]:
# SKIP

# Cleanup

In [ ]:
spark.stop()